In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import os
import time
from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score
from itertools import cycle
import numpy as np
from tqdm import tqdm
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import ReduceLROnPlateau
import warnings
import logging # Standard library logging
warnings.filterwarnings('ignore')

# --- Configuration ---
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("D:/Teeth Disease Classification Dental/backend/model/ViT_Oral_Daddy_2")
LOG_DIR = Path("D:/Teeth Disease Classification Dental/backend/model/ViT_Oral_Daddy_2")

# --- OPTIMIZED HYPERPARAMETERS ---
BATCH_SIZE = 32
FEATURE_EXTRACT_EPOCHS = 4
FINE_TUNE_EPOCHS = 20
LR_HEAD = 0.003
LR_FINETUNE = 1e-5
WEIGHT_DECAY = 1e-3
PATIENCE = 5
MIN_DELTA = 0.003
DROPOUT_HEAD = 0.5
DROPOUT_INTERMEDIATE = 0.4
INTERMEDIATE_SIZE = 512
GRAD_CLIP_NORM = 0.5
NUM_WORKERS = 0
FOCAL_LOSS_GAMMA = 2.0 # Focusing parameter for Focal Loss

# --- Device Setup ---
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()

# --- TQDM-aware Logging Handler ---
class TqdmLoggingHandler(logging.Handler):
    def __init__(self, level=logging.NOTSET):
        super().__init__(level)
    def emit(self, record):
        try:
            msg = self.format(record)
            tqdm.write(msg)
            self.flush()
        except (KeyboardInterrupt, SystemExit):
            raise
        except Exception:
            self.handleError(record)

# --- Logging Setup ---
def setup_logging():
    LOG_DIR.mkdir(exist_ok=True)
    log_filename = LOG_DIR / f"training_log_{time.strftime('%Y%m%d-%H%M%S')}.log"
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    formatter = logging.Formatter('[%(asctime)s] - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    
    file_handler = logging.FileHandler(log_filename, encoding='utf-8')
    file_handler.setFormatter(formatter)
    
    console_handler = TqdmLoggingHandler()
    console_handler.setFormatter(formatter)
    
    if not logger.handlers:
        logger.addHandler(file_handler)
        logger.addHandler(console_handler)
    return logger

# --- Enhanced Data Transforms ---
def get_transforms():
    train_transforms = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.88, 1.0), ratio=(0.9, 1.1)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
        transforms.RandomRotation(degrees=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.1, scale=(0.02, 0.08))
    ])
    val_test_transforms = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return {'train': train_transforms, 'val': val_test_transforms, 'test': val_test_transforms}

# --- Load Dataset ---
def load_data(logger):
    try:
        data_transforms = get_transforms()
        image_datasets = {x: datasets.ImageFolder(str(DATA_DIR / x), data_transforms[x]) for x in ['train', 'val', 'test']}
        dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'),
                                     num_workers=NUM_WORKERS, pin_memory=True, drop_last=(x == 'train'))
                       for x in ['train', 'val', 'test']}
        dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
        class_names = image_datasets['train'].classes
        logger.info(f"Data loaded successfully. Classes: {class_names}")
        logger.info(f"Dataset sizes: Train={dataset_sizes['train']}, Val={dataset_sizes['val']}, Test={dataset_sizes['test']}")
        return dataloaders, dataset_sizes, class_names
    except FileNotFoundError:
        logger.error(f"Data directory not found at '{DATA_DIR}'.")
        logger.error("Please ensure the dataset exists at the specified path.")
        exit(1)

# --- Advanced Early Stopping ---
class EarlyStopping:
    def __init__(self, patience=4, min_delta=0.003, restore_best_weights=True, verbose=True):
        self.patience, self.min_delta, self.restore_best_weights = patience, min_delta, restore_best_weights
        self.verbose, self.best_score, self.counter = verbose, None, 0
        self.best_weights, self.best_epoch = None, 0
    def __call__(self, val_score, model, epoch, logger):
        if self.best_score is None:
            self.best_score = val_score; self.save_checkpoint(model, epoch)
            return False
        if val_score > self.best_score + self.min_delta:
            self.best_score = val_score; self.save_checkpoint(model, epoch); self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                logger.info(f'EarlyStopping: {self.counter}/{self.patience}, best: {self.best_score:.4f} (epoch {self.best_epoch})')
        if self.counter >= self.patience:
            if self.restore_best_weights and self.best_weights:
                model.load_state_dict(self.best_weights)
                if self.verbose: logger.info(f'Restored best weights from epoch {self.best_epoch}')
            return True
        return False
    def save_checkpoint(self, model, epoch):
        self.best_weights = model.state_dict().copy(); self.best_epoch = epoch

# --- Enhanced Model Creation ---
def create_model(num_classes, logger):
    logger.info("Setting up Vision Transformer (vit_b_16)...")
    model = models.vit_b_16(weights='IMAGENET1K_V1')
    num_ftrs = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.LayerNorm(num_ftrs), nn.Dropout(p=DROPOUT_HEAD), nn.Linear(num_ftrs, INTERMEDIATE_SIZE),
        nn.GELU(), nn.Dropout(p=DROPOUT_INTERMEDIATE), nn.Linear(INTERMEDIATE_SIZE, num_classes)
    )
    return model.to(device)

# --- Training Function ---
def train_model(model, criterion, optimizer, scheduler, dataloaders, dataset_sizes, num_epochs, phase_name, logger, use_early_stopping=False):
    since = time.time()
    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'learning_rates': []}
    early_stopping = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA) if use_early_stopping else None
    scaler = GradScaler(enabled=use_amp)
    for epoch in range(num_epochs):
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss, running_corrects = 0.0, 0
            pbar_desc = f"Epoch {epoch+1}/{num_epochs} [{phase.capitalize():^5}] ({phase_name})"
            pbar = tqdm(dataloaders[phase], desc=pbar_desc, unit="batch", ncols=120)
            for inputs, labels in pbar:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.set_grad_enabled(phase == 'train'):
                    with autocast(device_type=device.type, enabled=use_amp):
                        outputs = model(inputs); loss = criterion(outputs, labels)
                    if phase == 'train':
                        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM); scaler.step(optimizer); scaler.update()
                _, preds = torch.max(outputs, 1)
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{(torch.sum(preds == labels.data) / inputs.size(0)):.3f}')
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            history[f'{phase}_loss'].append(epoch_loss); history[f'{phase}_acc'].append(epoch_acc.item())
            current_lr = optimizer.param_groups[0]['lr']
            logger.info(f"Epoch {epoch+1}/{num_epochs} [{phase.capitalize():^5}] -> Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, LR: {current_lr:.2e}")
            if phase == 'val':
                history['learning_rates'].append(current_lr)
                if isinstance(scheduler, ReduceLROnPlateau):
                    scheduler.step(epoch_loss)
                elif scheduler is not None:
                    scheduler.step()
                if epoch_acc > best_acc: best_acc = epoch_acc
                if early_stopping and early_stopping(epoch_acc, model, epoch, logger):
                    logger.info(f'Early stopping triggered at epoch {epoch+1}')
                    return model, history
    time_elapsed = time.time() - since
    logger.info(f'Phase "{phase_name}" completed in {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s')
    logger.info(f'Best validation accuracy for this phase: {best_acc:.4f}')
    return model, history

# --- Evaluation & Visualization Functions ---
def evaluate_model(model, dataloader):
    model.eval(); y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Evaluating Test Set", ncols=100):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs); probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy()); y_pred.extend(preds.cpu().numpy()); y_probs.extend(probs.cpu().numpy())
    return y_true, y_pred, y_probs

def save_misclassified_images(model, dataloader, class_names, num_images=15):
    misclassified_path = OUTPUT_DIR / "misclassified_samples"
    misclassified_path.mkdir(exist_ok=True)
    model.eval(); misclassified_count = 0
    random_dataloader = DataLoader(dataloader.dataset, batch_size=BATCH_SIZE, shuffle=True)
    mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).to(device)
    with torch.no_grad():
        for inputs, labels in random_dataloader:
            if misclassified_count >= num_images: break
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs); _, preds = torch.max(outputs, 1)
            errors = (preds != labels)
            for i in range(inputs.size(0)):
                if errors[i] and misclassified_count < num_images:
                    img = inputs[i].clone(); img = img * std[:, None, None] + mean[:, None, None]
                    img = transforms.ToPILImage()(img.cpu())
                    true_label, pred_label = class_names[labels[i]], class_names[preds[i]]
                    filename = misclassified_path / f"{misclassified_count+1}_true_{true_label}_pred_{pred_label}.png"
                    img.save(filename); misclassified_count += 1

def save_visualizations(history_list, y_true, y_pred, y_probs, class_names, logger):
    metrics_path = OUTPUT_DIR / "metrics"
    metrics_path.mkdir(exist_ok=True)
    y_true_bin = np.eye(len(class_names))[y_true]
    y_probs = np.array(y_probs)
    full_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for history in history_list:
        for key in full_history.keys():
            full_history[key].extend(history.get(key, []))
    plt.figure(figsize=(12, 5)); plt.subplot(1, 2, 1)
    plt.plot(full_history['train_loss'], label='Train Loss'); plt.plot(full_history['val_loss'], label='Validation Loss')
    plt.title('Training & Validation Loss'); plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(full_history['train_acc'], label='Train Accuracy'); plt.plot(full_history['val_acc'], label='Validation Accuracy')
    plt.title('Training & Validation Accuracy'); plt.xlabel('Epochs'); plt.ylabel('Accuracy'); plt.legend()
    plt.tight_layout(); plt.savefig(metrics_path / "loss_accuracy_curves.png", dpi=300); plt.close()
    cm = confusion_matrix(y_true, y_pred); plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix'); plt.xlabel('Predicted Label'); plt.ylabel('True Label')
    plt.tight_layout(); plt.savefig(metrics_path / "confusion_matrix.png", dpi=300); plt.close()
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(len(class_names))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(class_names)): mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= len(class_names)
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
    plt.figure(figsize=(12, 10)); colors = cycle(plt.cm.get_cmap('tab20').colors)
    plt.plot(fpr["micro"], tpr["micro"], label=f'Micro-average ROC (area = {roc_auc["micro"]:0.2f})', color='deeppink', linestyle=':', linewidth=4)
    plt.plot(fpr["macro"], tpr["macro"], label=f'Macro-average ROC (area = {roc_auc["macro"]:0.2f})', color='navy', linestyle=':', linewidth=4)
    for i, color in zip(range(len(class_names)), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'{class_names[i]} (area = {roc_auc[i]:0.2f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2); plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('Multi-class ROC Curves'); plt.legend(loc="lower right")
    plt.tight_layout(); plt.savefig(metrics_path / "roc_curves_enhanced.png", dpi=300); plt.close()
    precision, recall, avg_precision = {}, {}, {}
    for i in range(len(class_names)):
        precision[i], recall[i], _ = precision_recall_curve(y_true_bin[:, i], y_probs[:, i])
        avg_precision[i] = average_precision_score(y_true_bin[:, i], y_probs[:, i])
    plt.figure(figsize=(12, 10));
    for i, color in zip(range(len(class_names)), colors):
        plt.plot(recall[i], precision[i], color=color, lw=2, label=f'{class_names[i]} (AP = {avg_precision[i]:0.2f})')
    plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Multi-class Precision-Recall Curves'); plt.legend(loc="best")
    plt.tight_layout(); plt.savefig(metrics_path / "precision_recall_curves.png", dpi=300); plt.close()
    logger.info("All visualizations saved successfully.")

# --- Focal Loss Implementation ---
class FocalLoss(nn.Module):
    """
    Implements the Focal Loss from the paper "Focal Loss for Dense Object Detection".
    It is a dynamically scaled cross-entropy loss, where the scaling factor
    decays to zero as confidence in the correct class increases.
    """
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Calculate the cross-entropy loss, but without any reduction
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        
        # Calculate the probability of the correct class
        # pt = exp(-ce_loss)
        pt = torch.exp(-ce_loss)
        
        # This is the core of Focal Loss. It adds a modulating factor (1-pt)^gamma
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.alpha is not None:
            if not isinstance(self.alpha, torch.Tensor):
                self.alpha = torch.tensor(self.alpha, device=inputs.device)
            # Apply alpha weighting based on the target class
            alpha_t = self.alpha.gather(0, targets.data.view(-1))
            focal_loss = alpha_t * focal_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# --- Main Execution Flow ---
def main():
    logger = setup_logging()
    
    logger.info("╔════════════════════════════════════════════╗")
    logger.info("║   Oral Disease Classification Training   ║")
    logger.info("╚════════════════════════════════════════════╝")
    
    logger.info(f"Using device: {device}")
    if use_amp: logger.info("Automatic Mixed Precision (AMP): ENABLED")
    logger.info("\n" + "─"*20 + " HYPERPARAMETERS " + "─"*20)
    logger.info(f"Batch Size: {BATCH_SIZE}")
    logger.info(f"Feature Extract Epochs: {FEATURE_EXTRACT_EPOCHS}")
    logger.info(f"Fine-Tune Epochs: {FINE_TUNE_EPOCHS}")
    logger.info(f"Learning Rate (Head): {LR_HEAD}")
    logger.info(f"Learning Rate (Fine-Tune): {LR_FINETUNE}")
    logger.info(f"Weight Decay: {WEIGHT_DECAY}")
    logger.info(f"Optimizer: AdamW")
    logger.info(f"Loss Function: Focal Loss (gamma={FOCAL_LOSS_GAMMA})") # Log the new loss function
    logger.info("─"*57 + "\n")
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    dataloaders, dataset_sizes, class_names = load_data(logger)
    model = create_model(len(class_names), logger)
    
    # +++ MODIFICATION: Use Focal Loss instead of CrossEntropyLoss +++
    # Note: Label smoothing is generally not used with Focal Loss as their
    # objectives can conflict. We remove it.
    criterion = FocalLoss(gamma=FOCAL_LOSS_GAMMA)
    logger.info(f"Using Focal Loss with gamma={FOCAL_LOSS_GAMMA}")
    
    # PHASE 1
    logger.info("\n" + "═"*20 + " PHASE 1: FEATURE EXTRACTION " + "═"*18)
    for param in model.parameters(): param.requires_grad = False
    for param in model.heads.parameters(): param.requires_grad = True
    optimizer_head = optim.AdamW(model.heads.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    model, history1 = train_model(model, criterion, optimizer_head, None, dataloaders, dataset_sizes, FEATURE_EXTRACT_EPOCHS, "Feature Extraction", logger)
    
    # PHASE 2
    logger.info("\n" + "═"*23 + " PHASE 2: FINE-TUNING " + "═"*21)
    for param in model.parameters(): param.requires_grad = True
    optimizer_finetune = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
    scheduler_finetune = optim.lr_scheduler.CosineAnnealingLR(optimizer_finetune, T_max=FINE_TUNE_EPOCHS, eta_min=1e-7)
    model, history2 = train_model(model, criterion, optimizer_finetune, scheduler_finetune, dataloaders, dataset_sizes, FINE_TUNE_EPOCHS, "Fine-Tuning", logger, use_early_stopping=True)
    
    # EVALUATION
    logger.info("\n" + "═"*20 + " FINAL EVALUATION & REPORTING " + "═"*15)
    y_true, y_pred, y_probs = evaluate_model(model, dataloaders['test'])
    test_acc = np.mean(np.array(y_true) == np.array(y_pred))
    logger.info(f"Final Test Accuracy: {test_acc:.4f}")
    
    report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    logger.info("\n" + "Classification Report:\n" + report_str)
    
    # SAVE RESULTS & VISUALIZATIONS
    save_visualizations([history1, history2], y_true, y_pred, y_probs, class_names, logger)
    save_misclassified_images(model, dataloaders['test'], class_names)
    logger.info("Misclassified image samples saved.")

    torch.save({'model_state_dict': model.state_dict(), 'best_accuracy': max(history2.get('val_acc', [0])), 'class_names': class_names}, OUTPUT_DIR / "final_model.pth")
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    with open(OUTPUT_DIR / "classification_report.json", 'w') as f:
        json.dump(report_dict, f, indent=2)
    
    logger.info("\n" + "═"*28 + " SUMMARY " + "═"*28)
    logger.info(f"Training completed successfully!")
    logger.info(f"Model saved to:       {OUTPUT_DIR / 'final_model.pth'}")
    logger.info(f"Report saved to:      {OUTPUT_DIR / 'classification_report.json'}")
    logger.info(f"Visualizations saved to: {OUTPUT_DIR / 'metrics'}")
    logger.info(f"Logs saved to:        {LOG_DIR}")
    logger.info("═"*65)

if __name__ == '__main__':
    main()

[2025-09-06 18:39:43] - INFO - ╔════════════════════════════════════════════╗
[2025-09-06 18:39:43] - INFO - ║   Oral Disease Classification Training   ║
[2025-09-06 18:39:43] - INFO - ╚════════════════════════════════════════════╝
[2025-09-06 18:39:43] - INFO - Using device: cuda:0
[2025-09-06 18:39:43] - INFO - Automatic Mixed Precision (AMP): ENABLED
[2025-09-06 18:39:43] - INFO - 
──────────────────── HYPERPARAMETERS ────────────────────
[2025-09-06 18:39:43] - INFO - Batch Size: 32
[2025-09-06 18:39:43] - INFO - Feature Extract Epochs: 4
[2025-09-06 18:39:43] - INFO - Fine-Tune Epochs: 20
[2025-09-06 18:39:43] - INFO - Learning Rate (Head): 0.003
[2025-09-06 18:39:43] - INFO - Learning Rate (Fine-Tune): 1e-05
[2025-09-06 18:39:43] - INFO - Weight Decay: 0.001
[2025-09-06 18:39:43] - INFO - Optimizer: AdamW
[2025-09-06 18:39:43] - INFO - Loss Function: Focal Loss (gamma=2.0)
[2025-09-06 18:39:43] - INFO - ─────────────────────────────────────────────────────────

[2025-09-06 18:39:

Epoch 1/4 [Train] (Feature Extraction): 100%|██████████████| 137/137 [01:48<00:00,  1.26batch/s, acc=0.656, loss=0.5251]


[2025-09-06 18:41:33] - INFO - Epoch 1/4 [Train] -> Loss: 0.6896, Acc: 0.6006, LR: 3.00e-03


Epoch 1/4 [ Val ] (Feature Extraction): 100%|████████████████| 30/30 [00:13<00:00,  2.26batch/s, acc=0.333, loss=0.6071]


[2025-09-06 18:41:46] - INFO - Epoch 1/4 [ Val ] -> Loss: 0.4973, Acc: 0.6916, LR: 3.00e-03


...

[2025-09-06 19:03:55] - INFO - Epoch 12/20 [Train] -> Loss: 0.1131, Acc: 0.8657, LR: 4.28e-06


Epoch 12/20 [ Val ] (Fine-Tuning): 100%|█████████████████████| 30/30 [00:07<00:00,  3.92batch/s, acc=0.889, loss=0.2308]


[2025-09-06 19:04:02] - INFO - Epoch 12/20 [ Val ] -> Loss: 0.2445, Acc: 0.7972, LR: 4.28e-06
[2025-09-06 19:04:02] - INFO - EarlyStopping: 5/5, best: 0.8004 (epoch 6)
[2025-09-06 19:04:02] - INFO - Restored best weights from epoch 6
[2025-09-06 19:04:02] - INFO - Early stopping triggered at epoch 12
[2025-09-06 19:04:02] - INFO - 
════════════════════ FINAL EVALUATION & REPORTING ═══════════════


Evaluating Test Set: 100%|██████████████████████████████████████████| 30/30 [00:19<00:00,  1.56it/s]


[2025-09-06 19:04:21] - INFO - Final Test Accuracy: 0.8307
[2025-09-06 19:04:21] - INFO - 
Classification Report:
                      precision    recall  f1-score   support

            Calculus     0.7157    0.7449    0.7300       196
Cavity (Tooth Decay)     0.9375    0.9375    0.9375       144
           Fluorosis     0.6250    0.7692    0.6897        13
          Gingivitis     0.7809    0.7686    0.7747       255
       Healthy Teeth     0.8889    0.5714    0.6957        14
          Hypodontia     0.9115    0.8706    0.8906       201
         Mouth Ulcer     1.0000    0.9861    0.9930        72
      Spacing Issues     0.7333    0.9167    0.8148        12
 Tooth Discoloration     0.7755    0.8636    0.8172        44

            accuracy                         0.8307       951
           macro avg     0.8187    0.8254    0.8159       951
        weighted avg     0.8340    0.8307    0.8313       951

[2025-09-06 19:04:23] - INFO - All visualizations saved successfully.
[2025-0